# Phase 2 — Unsloth Smoke Test (Kaggle T4)

**Before running:** Settings (right panel) → Accelerator = **GPU T4 x2** (NOT P100 — no tensor cores), Internet = **On** (needs phone-verified account).

Four gate checks: (1) sec/step sane, (2) eval doesn't OOM, (3) checkpoint saves, (4) kill + resume works.
Budget: ~30–60 min of GPU. Stop the session the moment we're done.

In [ ]:
# CELL 1 — config. v1 fix #7: everything shell cells need goes through os.environ, HERE.
import os
os.environ["REPO_URL"]        = "https://github.com/Rick-Roy-JC/text-to-sql-qlora-v2.git"
os.environ["DATA_DATASET"]    = "/kaggle/input/spider-prepared-v2"   # your private Kaggle dataset
os.environ["CHECKPOINT_ROOT"] = "/kaggle/working/checkpoints"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth free tier = single GPU; hide the second T4

In [ ]:
# CELL 2 — sanity: which GPU did we actually get?
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
# Expect: Tesla T4, 15360 MiB, 7.5  (if you see P100 / compute_cap 6.0 -> STOP, change accelerator)

In [ ]:
# CELL 3 — install Unsloth (pulls compatible torch/trl/peft/bitsandbytes)
%pip install -q unsloth
import unsloth, trl, peft, transformers, torch
print("unsloth", unsloth.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__)
# PIN THESE: once the smoke test passes, copy this printout into notebooks/VERSIONS.txt
# so Phases 3-5 install identical versions (v1's peft/torchao clash, never again).

In [ ]:
# CELL 4 — clone repo, link prepared data
%cd /kaggle/working
!git clone $REPO_URL
%cd text-to-sql-qlora-v2
!mkdir -p data
!cp $DATA_DATASET/*.jsonl data/
!ls -la data/    # expect train.jsonl / val.jsonl / dev.jsonl directly here (no nesting!)

In [ ]:
# CELL 5 — SMOKE RUN: gate checks 1-3 (50 examples, 20 steps, eval+save at step 10)
!python src/train.py --smoke --output_root $CHECKPOINT_ROOT

In [ ]:
# CELL 6 — gate check 4a: simulated crash mid-run.
# 40-step run, killed after ~90 seconds — long enough to pass checkpoint-10/20.
import subprocess, time, signal
p = subprocess.Popen(["python", "src/train.py", "--smoke", "--max_steps", "40",
                      "--output_root", os.environ["CHECKPOINT_ROOT"]])
time.sleep(90)
p.send_signal(signal.SIGKILL)
p.wait()
print("killed. checkpoints on disk:")
!ls $CHECKPOINT_ROOT/r8-smoke/

In [ ]:
# CELL 7 — gate check 4b: resume. Must print '[resume] resuming from ...checkpoint-N'
# and finish the remaining steps WITHOUT a scaler.pt crash.
!python src/train.py --smoke --max_steps 40 --output_root $CHECKPOINT_ROOT

In [ ]:
# CELL 8 — verdict + cleanup reminder
!ls -la $CHECKPOINT_ROOT/r8-smoke/adapter_final/
print("If cells 5 and 7 both ended in PASS lines: Phase 2 gate MET.")
print("NOW: File -> Save Version (to keep logs), then STOP THIS SESSION. Quota is fuel.")